# Mini Demo: Python calcula el VaR, la IA ayuda a entender el riesgo

Demo conceptual de **stress testing aumentado por IA** para un portafolio simple de acciones.

**Tesis:** Python calcula el riesgo cuantitativo; la IA ayuda a interpretar, conectar escenarios y preparar mejores preguntas para el comité.

> Uso educativo. No es recomendación de inversión ni modelo regulatorio oficial.


## 1. Importación de librerías
En Colab, descomenta la instalación de `yfinance` si hace falta.

In [ ]:
# !pip install yfinance --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import yfinance as yf
import json

pd.options.display.float_format = '{:,.4f}'.format


## 2. Definición del portafolio

In [ ]:
tickers = ['SPY', 'NVDA', 'MSFT', 'AAPL']
weights = np.array([0.40, 0.25, 0.20, 0.15])
portfolio_value = 1_000_000  # USD

start_date = '2022-01-01'
end_date = None

confidence_levels = [0.95, 0.99]
horizons = [1, 10]

if len(tickers) != len(weights):
    raise ValueError('El número de tickers debe coincidir con el número de pesos.')
if not np.isclose(weights.sum(), 1.0):
    raise ValueError('Los pesos deben sumar 1.0.')

portfolio = pd.DataFrame({
    'Ticker': tickers,
    'Peso': weights,
    'Monto_USD': weights * portfolio_value
})
portfolio


## 3. Descarga de precios desde Yahoo Finance

In [ ]:
raw_data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True, progress=False)

if isinstance(raw_data.columns, pd.MultiIndex):
    prices = raw_data['Close'].copy()
else:
    prices = raw_data.copy()

prices = prices.dropna()
print(f'Observaciones descargadas: {len(prices)}')
prices.tail()


## 4. Retornos y métricas básicas

In [ ]:
returns = prices.pct_change().dropna()
portfolio_returns = returns @ weights

summary_assets = pd.DataFrame({
    'Retorno diario medio': returns.mean(),
    'Volatilidad diaria': returns.std(),
    'Volatilidad anualizada': returns.std() * np.sqrt(252)
})

mean_p = portfolio_returns.mean()
vol_p = portfolio_returns.std()

portfolio_summary = pd.DataFrame({
    'Métrica': ['Retorno diario medio', 'Volatilidad diaria', 'Volatilidad anualizada'],
    'Portafolio': [mean_p, vol_p, vol_p * np.sqrt(252)]
})

display(summary_assets)
display(portfolio_summary)


## 5. Correlación de activos

In [ ]:
corr = returns.corr()
display(corr)

plt.figure(figsize=(7, 5))
plt.imshow(corr, interpolation='nearest')
plt.xticks(range(len(tickers)), tickers)
plt.yticks(range(len(tickers)), tickers)
plt.colorbar(label='Correlación')
plt.title('Matriz de correlación de retornos')
plt.tight_layout()
plt.show()


## 6. VaR histórico y Expected Shortfall

El VaR histórico usa la distribución observada de retornos pasados.  
El Expected Shortfall mide la pérdida promedio más allá del umbral de VaR.


In [ ]:
def historical_var(returns_series, confidence_level, portfolio_value, horizon=1):
    alpha = 1 - confidence_level
    var_return = -np.quantile(returns_series, alpha)
    return var_return * np.sqrt(horizon) * portfolio_value

def historical_es(returns_series, confidence_level, portfolio_value, horizon=1):
    alpha = 1 - confidence_level
    threshold = np.quantile(returns_series, alpha)
    tail_losses = returns_series[returns_series <= threshold]
    es_return = -tail_losses.mean()
    return es_return * np.sqrt(horizon) * portfolio_value

hist_results = []
for cl in confidence_levels:
    for h in horizons:
        hist_results.append({
            'Método': 'Histórico',
            'Confianza': f'{int(cl*100)}%',
            'Horizonte_días': h,
            'VaR_USD': historical_var(portfolio_returns, cl, portfolio_value, h),
            'Expected_Shortfall_USD': historical_es(portfolio_returns, cl, portfolio_value, h)
        })

hist_df = pd.DataFrame(hist_results)
hist_df


## 7. VaR paramétrico normal

Asume distribución normal de retornos. Es útil como aproximación inicial, pero puede subestimar colas gruesas o eventos extremos.


In [ ]:
def parametric_var_normal(mean_return, vol_return, confidence_level, portfolio_value, horizon=1):
    alpha = 1 - confidence_level
    z = norm.ppf(alpha)
    var_return = -(mean_return * horizon + z * vol_return * np.sqrt(horizon))
    return var_return * portfolio_value

def parametric_es_normal(mean_return, vol_return, confidence_level, portfolio_value, horizon=1):
    alpha = 1 - confidence_level
    z = norm.ppf(alpha)
    es_return = -(mean_return * horizon - vol_return * np.sqrt(horizon) * norm.pdf(z) / alpha)
    return es_return * portfolio_value

param_results = []
for cl in confidence_levels:
    for h in horizons:
        param_results.append({
            'Método': 'Paramétrico Normal',
            'Confianza': f'{int(cl*100)}%',
            'Horizonte_días': h,
            'VaR_USD': parametric_var_normal(mean_p, vol_p, cl, portfolio_value, h),
            'Expected_Shortfall_USD': parametric_es_normal(mean_p, vol_p, cl, portfolio_value, h)
        })

param_df = pd.DataFrame(param_results)
param_df


## 8. Consolidado de resultados

In [ ]:
risk_results = pd.concat([hist_df, param_df], ignore_index=True)
risk_results['VaR_%_Portafolio'] = risk_results['VaR_USD'] / portfolio_value
risk_results['ES_%_Portafolio'] = risk_results['Expected_Shortfall_USD'] / portfolio_value
risk_results


## 9. Gráfico de distribución de retornos

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(portfolio_returns, bins=50, alpha=0.75)

for cl in confidence_levels:
    var_threshold = np.quantile(portfolio_returns, 1 - cl)
    plt.axvline(var_threshold, linestyle='--', label=f'VaR histórico {int(cl*100)}%')

plt.title('Distribución histórica de retornos diarios del portafolio')
plt.xlabel('Retorno diario')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.show()


## 10. Stress testing simple

In [ ]:
stress_scenarios = {
    'Caída leve (-3%)': -0.03,
    'Caída moderada (-7%)': -0.07,
    'Caída severa (-12%)': -0.12,
    'Caída extrema (-20%)': -0.20,
}

stress_df = pd.DataFrame([
    {
        'Escenario': name,
        'Shock_portafolio': shock,
        'Pérdida_USD': -shock * portfolio_value,
        'Valor_post_stress_USD': portfolio_value * (1 + shock)
    }
    for name, shock in stress_scenarios.items()
])
stress_df


## 11. Resumen para el GPT/copiloto

Esta salida se copia y pega en el GPT para que genere interpretación ejecutiva, escenarios narrativos y preguntas de comité.


In [ ]:
summary_for_gpt = {
    'objetivo': 'Interpretar resultados de VaR y stress testing para un comité de riesgos.',
    'advertencia': 'Demo educativa. No es recomendación de inversión ni modelo regulatorio oficial.',
    'portafolio': portfolio.to_dict(orient='records'),
    'periodo_precios': {
        'inicio': str(prices.index.min().date()),
        'fin': str(prices.index.max().date()),
        'observaciones': int(len(prices))
    },
    'metricas_portafolio': {
        'retorno_diario_medio': float(mean_p),
        'volatilidad_diaria': float(vol_p),
        'volatilidad_anualizada': float(vol_p * np.sqrt(252))
    },
    'resultados_var_es': risk_results.to_dict(orient='records'),
    'stress_tests': stress_df.to_dict(orient='records'),
    'correlacion': corr.round(4).to_dict(),
    'instruccion_para_gpt': {
        'rol': 'Actúa como copiloto de escenarios de riesgo financiero.',
        'tareas': [
            'Explicar los resultados en lenguaje ejecutivo.',
            'Identificar principales fuentes de riesgo del portafolio.',
            'Sugerir escenarios narrativos que podrían no estar capturados por el VaR.',
            'Proponer señales tempranas a monitorear.',
            'Preparar preguntas para un comité de riesgos.',
            'Indicar limitaciones del VaR y del análisis histórico.',
            'Sugerir qué recalcular o validar en Python.'
        ],
        'restricciones': [
            'No recomendar comprar o vender activos.',
            'No presentar escenarios como predicciones.',
            'No reemplazar juicio experto ni validación independiente.',
            'Separar resultados cuantitativos de interpretación cualitativa.'
        ]
    }
}

print(json.dumps(summary_for_gpt, indent=2, ensure_ascii=False))


## 12. Prompt sugerido para el GPT/copiloto

Copia el JSON anterior y pégalo debajo de este prompt:

```text
Actúa como copiloto de escenarios de riesgo financiero.

Tu función no es reemplazar al analista ni al modelo cuantitativo. Tu función es interpretar resultados de VaR/stress testing, identificar riesgos no evidentes, sugerir escenarios narrativos, proponer señales tempranas a monitorear y preparar preguntas para un comité de riesgos.

Usa el JSON de resultados que te comparto como fuente principal. No inventes cifras. No recomiendes comprar o vender activos. Separa claramente:

1. Lectura cuantitativa.
2. Interpretación ejecutiva.
3. Escenarios narrativos no capturados por el VaR.
4. Señales tempranas a monitorear.
5. Preguntas para el comité.
6. Limitaciones del análisis.
7. Qué debería recalcularse o validarse en Python.

Mantén tono ejecutivo, técnico y pedagógico.
```
